In [1]:
import tskit
import pandas as pd
from lets_plot import *
LetsPlot.setup_html()
import numpy as np
from collections import Counter
import allel
import os

chromosome = 16

In [2]:
# Check the ones that are mispolarised - ONLY the mispolarised
chromosomes = [14, 15, 16]
misPolSfs = pd.DataFrame()
for chromosome in chromosomes:
    # Read in the vcf
    vcf_file = "/home/jobste/Data/Remapped/Chr" + str(chromosome) + "_remapped_NonHybrid.vcf"
    vcf = allel.read_vcf(vcf_file)
    gno = allel.GenotypeArray(vcf['calldata/GT'])
    pos = vcf['variants/POS']

    # Get a list of mutations files
    mutsFiles = [x for x in os.listdir("/home/jobste/Singer/Chr" + str(chromosome)) if "mut" in x]

    # Get mispolarised SNPs
    misPol = pd.DataFrame()
    numSites = 0
    for file in mutsFiles:
        chunkStart = int(file.split("_")[2]) * 1e6
        chunkEnd = int(file.split("_")[3]) * 1e6
        tmp = pd.read_table("Chr" + str(chromosome) + "/" + file, header=None, sep=" ", names=["Pos", "X", "ParentNode", "Der"])
        numSites += len(tmp)
        misPolTmp = tmp[(tmp.ParentNode == -1) & (tmp.Der == 1)]
        misPolTmp.loc[:, "POS"] = chunkStart + misPolTmp.Pos
        misPolTmp.loc[:, "Chr"] = chromosome
        misPolTmp.loc[:, "Chunk"] = str(chunkStart)
        s = set(pos)
        misPol = pd.concat([misPol, misPolTmp])


    frq = gno.count_alleles()
    is_biall = frq.is_biallelic()


    frqD = frq[is_biall][:,1] 
    frqA = frq[is_biall][:,0] 
    pos = pos[is_biall]
    misPolSet = set(misPol.POS.to_list())
    misPolBoolean = [x in misPolSet for x in pos]


    posMisPol = pos[misPolBoolean]
    frqDMisPol = frqD[misPolBoolean]
    misPolSfsTmp = pd.DataFrame.from_dict(Counter(frqDMisPol), "index").reset_index()
    misPolSfsTmp.columns = ["Num", "Count"]
    misPolSfsTmp.loc[:, "LogCount"] = np.log(misPolSfsTmp.Count)
    misPolSfsTmp.loc[:, "Chr"] = chromosome
    misPolSfs = pd.concat([misPolSfs, misPolSfsTmp])





/tmp/ipykernel_568263/1605026429.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  misPolTmp.loc[:, "POS"] = chunkStart + misPolTmp.Pos
/tmp/ipykernel_568263/1605026429.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  misPolTmp.loc[:, "Chr"] = chromosome
/tmp/ipykernel_568263/1605026429.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.o

In [3]:
ggplot(data = misPolSfs) + geom_point(aes(x = "Num", y = "Count")) + facet_wrap(facets = "Chr")

In [29]:
len(misPol) / numSites

0.07425850086870191

In [ ]:
frq = gno.count_alleles()
is_biall = frq.is_biallelic()



In [ ]:
frqD = frq[is_biall][:,1] 
frqA = frq[is_biall][:,0] 
pos = pos[is_biall]

In [ ]:
misPolSet = set(misPol.POS.to_list())
misPolBoolean = [x in misPolSet for x in pos]


In [ ]:
sum(misPolBoolean)

19148

In [ ]:
posMisPol = pos[misPolBoolean]
frqDMisPol = frqD[misPolBoolean]
misPolSfs = pd.DataFrame.from_dict(Counter(frqDMisPol), "index").reset_index()
misPolSfs.columns = ["Num", "Count"]
misPolSfs.loc[:, "LogCount"] = np.log(misPolSfs.Count)
ggplot(data = misPolSfs) + geom_point(aes(x = "Num", y = "Count"))

In [ ]:
#OLD CHECKING - not correct
# CHECK POLARISATION!
mut_freq = {}
remove_sites = []
for t in ts.trees():
    for s in t.sites():
        muts = s.mutations
        if len(muts) == 1:
            node_below = muts[0].node
            assert not s.position in mut_freq
            mut_freq[chunkStart + s.position] = t.num_samples(node_below)
        else:
            remove_sites.append(s.id)



import allel
vcf_file = "/home/jobste/Data/Remapped/Chr" + str(chromosome) + "_remapped_NonHybrid.vcf"
vcf = allel.read_vcf(vcf_file)
gno = allel.GenotypeArray(vcf['calldata/GT'])
pos = vcf['variants/POS']
print(len(vcf['samples']), ts.num_samples)




frq = gno.count_alleles()
is_biall = frq.is_biallelic()
frqD = frq[is_biall][:,1] 
frqA = frq[is_biall][:,0] 
pos = pos[is_biall]


chunkSites = [((x > chunkStart) & (x < chunkEnd)) for x in pos] #
frqD = frqD[chunkSites]
frqA = frqA[chunkSites]
pos = pos[chunkSites]

#derived count
mut_freq_vcfD = {p:f for p, f in zip(pos,frqD)}
mut_freq_vcfA = {p:f for p, f in zip(pos,frqA)}

for p, f in mut_freq.items():
    if p in mut_freq_vcfD:
        f2 = mut_freq_vcfD[p]
        #if f2 != f: print(p, f, f2)



mut_freq_df = pd.DataFrame.from_dict(mut_freq, "index").reset_index()
mut_freq_df.columns = ["Pos", "Count_Singer"]
mut_freq_vcf_dfD = pd.DataFrame.from_dict(mut_freq_vcfD, "index").reset_index()
mut_freq_vcf_dfD.columns = ["Pos", "Count_Vcf_Der"]
mut_freq_vcf_dfA = pd.DataFrame.from_dict(mut_freq_vcfA, "index").reset_index()
mut_freq_vcf_dfA.columns = ["Pos", "Count_Vcf_Anc"]

mut_freq_all_df = mut_freq_df.merge(mut_freq_vcf_dfD, how = "left", on = "Pos")
mut_freq_all_df = mut_freq_all_df.merge(mut_freq_vcf_dfA, how = "left", on = "Pos")


mut_freq_all_df.head()
mut_freq_all_df.loc[:, "Diff"] = mut_freq_all_df.Count_Vcf_Der - mut_freq_all_df.Count_Singer
mut_freq_all_df.loc[:, "Sum"] = mut_freq_all_df.Count_Vcf_Der + mut_freq_all_df.Count_Singer
mut_freq_all_df.loc[:, "SumVcf"] = mut_freq_all_df.Count_Vcf_Der + mut_freq_all_df.Count_Vcf_Anc
ggplot(data = mut_freq_all_df) + geom_histogram(aes(x = "Diff"))
# Check the ones that are mispolarised
muts = pd.read_table("Chr" + str(chromosome) + "_NonH_Chunk" + chunk + "/Chr" + str(chromosome) + "_NonHybrid_" + chunk + "_muts_99.txt", header=None, sep=" ", names=["Pos", "X", "ParentNode", "Der"])
misPol = muts[(muts.ParentNode == -1) & (muts.Der == 1)]
misPol.loc[:, "POS"] = chunkStart + misPol.Pos
gno = allel.GenotypeArray(vcf['calldata/GT'])
pos = vcf['variants/POS']

frq = gno.count_alleles()
is_biall = frq.is_biallelic()
frqD = frq[is_biall][:,1] 
frqA = frq[is_biall][:,0] 
pos = pos[is_biall]
misPolBoolean = [x in set(misPol.POS.to_list()) for x in pos]
posMisPol = pos[misPolBoolean]
frqDMisPol = frqD[misPolBoolean]
misPolSfs = pd.DataFrame.from_dict(Counter(frqDMisPol), "index").reset_index()
misPolSfs.columns = ["Num", "Count"]
misPolSfs.loc[:, "LogCount"] = np.log(misPolSfs.Count)
ggplot(data = misPolSfs) + geom_point(aes(x = "Num", y = "Count"))

In [ ]:


# Check for the uniquely mapped sites
ts = ts.delete_sites(remove_sites)
afs_Singer_unfold = ts.allele_frequency_spectrum(polarised = True, span_normalise = False)
afs_Singer_fold = ts.allele_frequency_spectrum(polarised = False, span_normalise = False)
afsSinger_fold_Df = pd.DataFrame({"Num": range(1, len(afs_Singer_fold)+1), "Count": np.log(afs_Singer_fold)})
ggplot(data = afsSinger_fold_Df) + geom_point(aes(x = "Num", y = "Count"))
afsSinger_unfold_Df = pd.DataFrame({"Num": range(1, len(afs_Singer_unfold)+1), "Count": np.log(afs_Singer_unfold)})
ggplot(data = afsSinger_unfold_Df) + geom_point(aes(x = "Num", y = "Count"))
keptPos = [7000000 + x.position for x in ts.sites()]
keptVcfPos = [x in keptPos for x in pos]
frqDKept = frqD[keptVcfPos]
frqAKept = frqA[keptVcfPos]
posKept = pos[keptVcfPos]
#derived count
mut_freq_vcfDKept = {p:f for p, f in zip(posKept,frqDKept)}
mut_freq_vcfAKept = {p:f for p, f in zip(posKept,frqAKept)}


vcfKeptSfs = pd.DataFrame.from_dict(Counter(mut_freq_vcfDKept.values()), "index").reset_index()
vcfKeptSfs.columns = ["Num", "Count"]
vcfKeptSfs.loc[:, "LogCount"] = np.log(vcfKeptSfs.Count)

ggplot(data = vcfKeptSfs) + geom_point(aes(x = "Num", y = "LogCount"))